This notebook is designed to determine the base window for neutron detection.
We use data from a reference experiment with sufficient neutron signal (currently ID-419).
The PSD/Energy data is then put into a 2D histogram, with energy cuts approximately every 15 keVee, and X total PSD cuts.
This 2D histogram is split using the energy cuts into a series of 1D histogram "energy" slices.
Each slice is modeled on a bimodal distribution of PSD vs. count, with the lower PSD Gaussian being for gamma ray events, and the higher PSD Gaussian being for neutrons.
Using these fit values, the FOM value is also calculated as abs(mu_n-mu_g)/(2.56*(sigma_n+sigma_g))
These values are used to define the neutron window as follows:

- Left boundary: Find energy A where FOM passes 1.27
- Right boundary: E = 688 keVee (Compton scattering edge, adjusted by detector energy resolution)
- Bottom boundary: For each slice, get point where x = slice energy midpoint, y = mu_n - 3 * sigma_n, connect points
- Top boundary: For each slice, get point where x = slice energy midpoint, y = mu_n + 3 * sigma_n, connect points

## Initialization

In [ ]:
# Importing needed code

from enum import Enum
from typing import Callable, TypeVar, Any
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from data_processing.arc_paths import (
    get_parq_root, get_exp_root, INPUT_DATA_FOLDER)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import ExperimentDataKey
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import calculate_timetag_hours
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import (
    generate_new_neutron_window,
    generate_nasa_neutron_window,
    classify
)
from data_processing.reporting.plotting import plot_classification
from data_processing.helpers.stop_jupyter import stop

In [ ]:
# Constants
SMOOTHING_WINDOW_SIZE = 5

In [ ]:
T = TypeVar('T')


def get_input_with_default(
    prompt: str, default: int, converter: Callable[[str], T]
) -> int:
    raw_value = input(prompt)
    try:
        value = converter(raw_value)
    except ValueError:
        value = default
    return value

In [ ]:
def get_window_size(data_type: str) -> int:
    prompt = (f"Enter smoothing window size for {data_type}, "
              + "or press Enter for default (5):")
    return get_input_with_default(prompt, SMOOTHING_WINDOW_SIZE, int)


def load_non_neutron_data(exp_name, file_names):
    if not isinstance(file_names, list):
        file_names = [file_names]
    for file_name in file_names:
        file_path = get_exp_root(exp_name) / file_name
        if file_path.is_file():
            df = pd.read_csv(file_path)
            return df
    return None


def smooth_non_neutron_data(
    df, raw_data_col_name, smoothed_data_col_name, window_size
):
    df[smoothed_data_col_name] = df[raw_data_col_name].rolling(
        window=window_size, min_periods=1).mean()
    return df


def localize_time(df, time_column):
    local_tz = 'America/Vancouver'
    naive_time = pd.to_datetime(df[time_column])
    try:
        localized_time = naive_time.dt.tz_localize(local_tz)
    except TypeError:
        localized_time = naive_time.dt.tz_convert(local_tz)
    df[time_column] = localized_time
    return df

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    zeroed_midpoint = pd.to_datetime(df["Bin midpoint"]) - start_time
    df['Bin time (s)'] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df['Time Bin'] = timetag_cut
    return df

In [ ]:
def get_endings_with_modifications(endings, string_method):
    modded_endings = [
        *endings,
        *[getattr(ending, string_method)() for ending in endings]
    ]
    return modded_endings


def get_possible_names(exp_name, endings):
    possible_names = [f"{exp_name}{ending}.csv"
                      for ending in possible_filename_endings]
    return possible_names


def get_filenames_with_hyphenless_ids(exp_name, filenames):
    hyphenless = exp_name.replace('-', '')
    modded_filenames = [
        *filenames,
        *[filename.replace(exp_name, hyphenless) for filename in filenames]
    ]
    return modded_filenames


def get_full_possible_names_set(exp_name, endings):
    possible_names = get_possible_names(exp_name, endings)
    full_poss_names_set = get_filenames_with_hyphenless_ids(
        exp_name, possible_names)
    return full_poss_names_set

In [ ]:
def find_failed_slices(
    df: pd.DataFrame,
    nan_total_threshold: int = 5,  # max bad slice fits total
    nan_window_threshold: int = 4,  # max bad slice fits in a "window"
    nan_rolling_window: int = 7  # window size
) -> tuple[pd.DataFrame, np.ndarray | None]:
    bad_slice_indexes = None
    nan_rows = df.isna().any(axis=1)
    nan_rows = nan_rows[nan_rows]
    if nan_rows.shape[0] > 0:
        nan_indexes = np.where(nan_rows)[0]
        total_nan_rows = len(nan_indexes)
        rolling_nan_count = nan_rows.rolling(window=nan_rolling_window) \
            .sum() \
            .max()

        print(f"Fit issues in {experiment_id}")
        print(f"Fit failed on following slice indexes: {nan_indexes}")

        if (total_nan_rows > nan_total_threshold
                or rolling_nan_count > nan_window_threshold):
            print(f"Experiment {experiment_id} could not be classified")
            print(f"Total failed slices: {total_nan_rows}")
            print(
                f"Max failed slices in a {nan_rolling_window} slice window:"+
                f" {rolling_nan_count}"
            )

            df = df.dropna().copy()
            bad_slice_indexes = nan_indexes

        # filter out all nan rows from df
        df = df.dropna().copy()
        # continue as normal to try fitting with bad rows ignored
    return df, bad_slice_indexes

## Experiment ID Input

In [ ]:
experiment_id = "ID-419"
# experiment_id = "ID-463"
id_valid = get_parq_root(experiment_id).is_dir()
if id_valid:
    print(f"Experiment {experiment_id} found")
else:
    print(f"Experiment {experiment_id} cannot be found")
    stop()
# done = False
# experiment_ids = ["ID-213"]
# while not done:
#     ids_valid = []
#     for exp_id in experiment_ids:
#         id_valid = get_parq_root(exp_id).is_dir()
#         ids_valid.append(id_valid)
#         if not id_valid:
#             print(f"Experiment {exp_id} cannot be found")

#     done = all(ids_valid)
#     if not done:
#         print("Invalid experiment IDs, please fix")
#         experiment_ids = []
#     else:
#         print("All experiment IDs are valid")

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
# Data Loading
exp_data: dict[str, Any] = {ExperimentDataKey.UNCLASSIFIED: load_psd(experiment_id)}

In [ ]:
# Express timetags in hours elapsed
exp_data[ExperimentDataKey.UNCLASSIFIED] = calculate_timetag_hours(exp_data[ExperimentDataKey.UNCLASSIFIED])

In [ ]:
# Recalibrate data
exp_data[ExperimentDataKey.UNCLASSIFIED] = recalibrate(exp_data[ExperimentDataKey.UNCLASSIFIED], Detector.ZERO)

In [ ]:
# Get PSD/Energy 2D histogram
start_scan_idx = 0
end_scan_idx = 420
energy_width = 5e-3

psd_report: pd.DataFrame = exp_data[ExperimentDataKey.UNCLASSIFIED]

calib_columns = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]

for calib_column, calib_key in zip(calib_columns, calib_keys):
    Z, xe, ye = get_psd_energy_histogram(
        psd_report, 
        calib_column,
        energy_width=energy_width
    )
    print(Z.shape)
    
    exp_data[calib_key] = {
        ExperimentDataKey.PSD_HISTOGRAM: Z,
        ExperimentDataKey.HISTOGRAM_X_EDGES: xe,
        ExperimentDataKey.HISTOGRAM_Y_EDGES: ye,
        ExperimentDataKey.END_SCAN_IDX: min(end_scan_idx, len(Z))
    }

In [ ]:
df = exp_data[ExperimentDataKey.UNCLASSIFIED]
# bg_time_limit = 5/60
# df = df.loc[df['TIMETAG_HOURS'] < bg_time_limit]
df

In [ ]:
# fig, ax = plt.subplots()
# ax.hist(df[DetectorDataframeColumn.RECALIBRATED_ENERGY.value], bins=100, alpha=0.2)
# # ax.set_xlim(1, 1.4)
# # ax.set_ylim(0, 20000)
# plt.hist(df[DetectorDataframeColumn.CALIB_ENERGY.value], bins=100, alpha=0.2)
# # plt.hist(df[DetectorDataframeColumn.ENERGY.value], bins=100, alpha=0.2)
# plt.show()

In [ ]:
# Scan energy slices, get bimodal fits
stop_here = False

caen_calibrated_data = exp_data[ExperimentDataKey.CAEN_CALIBRATION]
new_calibrated_data = exp_data[ExperimentDataKey.NEW_CALIBRATION]

for calib_key in [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]:
    calibrated_data = exp_data[calib_key]
    Z = calibrated_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = calibrated_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = calibrated_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = calibrated_data[ExperimentDataKey.END_SCAN_IDX]
    
    # Default
    default_bounds: BimodalBounds = (
        BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
        BimodalParams(0.2, 0.1, Z.max(),
            0.38, 0.04, 4000)
    )
    
    bounds_a: BimodalBounds = (
        BimodalParams(0.1, 0.01, 1,
            0.34, 0.01, 0),
        BimodalParams(0.2, 0.1, Z.max(),
            0.36, 0.04, 4000)
    )
    
    bounds_b: BimodalBounds = (
        BimodalParams(0.1, 0.01, 1,
            0.34, 0.01, 0),
        BimodalParams(0.2, 0.1, Z.max(),
            0.36, 0.03, 4000)
    )
    
    # Ranged Example
    bounds = [
        ((0, 60), bounds_a),
    ]
    
    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df)
    if bad_slice_indexes is not None:
        calibrated_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        calibrated_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        calibrated_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# # histogram contour plot (vaporwave island)
# cmap = plt.colormaps["nipy_spectral"]
# figsize = (24, 24)
# fontsize = 16
# histo_res = 128
# contour_res = 100
# angle_elev = 45
# angle_rot = 45

# fig = plt.figure(figsize=figsize)
# keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
# for i, calib_key in enumerate(keys):
#     data_dict = exp_data[calib_key]
#     Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
#     xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
#     ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
#     xlo = xe[:-1]
#     xhi = xe[1:]
#     ylo = ye[:-1]
#     yhi = ye[1:]
#     xmid = (xe[1:]+xe[:-1])/2
#     ymid = (ye[1:]+ye[:-1])/2
    
#     # ax = plt.axes(projection='3d')
#     ax = fig.add_subplot(len(keys), 1, i+1, projection='3d')
#     x, y = np.meshgrid(xmid, ymid)

#     ax.view_init(angle_elev, angle_rot)
#     ax.contour3D(x, y, Z.T, contour_res, cmap=cmap, alpha=0.6)
#     # ax.plot_surface(x, y, Z.T, cmap=cmap, alpha=0.6, rstride=1, cstride=10)
#     ax.contourf(x, y, Z.T, zdir='x', offset=2.6, cmap=cmap)
#     ax.contourf(x, y, Z.T, zdir='y', offset=-0.1, cmap=cmap)
#     ax.set(xlim=(-0.1, 2.6), ylim=(-0.1, 0.6))
#     # xpos = x.flatten()
#     # ypos = y.flatten()
#     # zpos = np.zeros_like(xpos)
#     # dx = xe[1]-xe[0]
#     # dy = ye[1]-ye[0]
#     # dz = Z.T.flatten()
#     # ax.bar3d(xpos, ypos, zpos, dx, dy, dz, cmap=cmap)
#     ax.set_title(f"PSD/Energy 3D Histogram - {calib_key.value}",
#                  fontsize=fontsize+4)
#     ax.set_ylabel("PSD", fontsize=fontsize)
#     ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
#     ax.set_zlabel("Counts", fontsize=fontsize)

# plt.show()

In [ ]:
# # Graph FOM threshold slice
# from data_processing.processing.figure_of_merit import bimodal
# slice_idx=30

# data_dict = exp_data[ExperimentDataKey.CAEN_CALIBRATION]
# xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
# ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
# Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
# df = data_dict[ExperimentDataKey.FOM_RESULTS]
# slice_energy = xe[slice_idx]
# ylo = ye[:-1]
# yhi = ye[1:]
# ymid = (ylo+yhi)/2
# histo_slice = Z.T[:, slice_idx]
# fom_slice = df.loc[slice_idx]
# fom = fom_slice[SliceFitDataframeColumn.FOM.value]
# params = tuple(fom_slice.iloc[1:7])

# fig, ax = plt.subplots(figsize=(8,8))
# ax.plot(ymid, histo_slice, "k", lw=4)
# ax.plot(ymid, bimodal(ylo, *params), "r--", lw=4)
# # ax.set_ylim(1, 6e3)
# ax.grid()
# # ax.set_yscale("log")
# ax.set_ylabel("Counts", fontsize=fontsize)
# ax.set_xlabel("PSD", fontsize=fontsize)
# ax.set_title(f"E={slice_energy:.3f} MeVee; FOM={fom:.3f}", fontsize=fontsize)
# ax.tick_params(axis='both', which='major', labelsize=fontsize)
# ax.tick_params(axis='both', which='minor', labelsize=fontsize)

In [ ]:
# Generate neutron window
sigma = 3  # Hey y'all, here's where you change sigma!

for calib_key in [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]:
    calib_data = exp_data[calib_key]
    df = calib_data[ExperimentDataKey.FOM_RESULTS]

    classic_nasa_borders = generate_nasa_neutron_window(df)
    recalc_nasa_borders = generate_nasa_neutron_window(df, recalculate_lower_energy_bound=True)
    new_borders = generate_new_neutron_window(df, sigma=sigma)
    calib_data[ExperimentDataKey.NASA_BORDERS] = classic_nasa_borders
    calib_data[ExperimentDataKey.NASA_BORDERS_RECALC] = recalc_nasa_borders
    calib_data[ExperimentDataKey.N_WINDOW_BORDERS] = new_borders

In [ ]:
# classify neutrons
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
energy_column_keys = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
border_keys = [ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC, ExperimentDataKey.N_WINDOW_BORDERS]
n_column_keys = [DetectorDataframeColumn.NEUTRON_CLASS, DetectorDataframeColumn.NEUTRON_RECALC_CLASS, DetectorDataframeColumn.NEW_N_CLASS]

for calib_key, en_col_key in zip(calib_keys, energy_column_keys):
    calib_data = exp_data[calib_key]
    calib_psd_report = psd_report.copy()
    for border_key, n_col_key in zip(border_keys, n_column_keys):
        borders = calib_data[border_key]
        calib_psd_report = classify(calib_psd_report, en_col_key, borders, n_col_key)
    calib_data[ExperimentDataKey.PSD_REPORT] = calib_psd_report

In [ ]:
psd_report_caen = exp_data[ExperimentDataKey.CAEN_CALIBRATION][ExperimentDataKey.PSD_REPORT]
psd_report_new = exp_data[ExperimentDataKey.NEW_CALIBRATION][ExperimentDataKey.PSD_REPORT]
print(psd_report_caen[psd_report_caen[DetectorDataframeColumn.NEUTRON_CLASS.value] == True].shape)
print(psd_report_new[psd_report_new[DetectorDataframeColumn.NEUTRON_CLASS.value] == True].shape)

## Export and Display

In [ ]:
# Plot classification
HISTOGRAM_RES = 1024
COUNT_LIMIT = 20

calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
energy_column_keys = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
border_keys = [
    ExperimentDataKey.NASA_BORDERS,
    ExperimentDataKey.NASA_BORDERS_RECALC,
    ExperimentDataKey.N_WINDOW_BORDERS
]
n_column_keys = [
    DetectorDataframeColumn.NEUTRON_CLASS, 
    DetectorDataframeColumn.NEUTRON_RECALC_CLASS, 
    DetectorDataframeColumn.NEW_N_CLASS
]

for calib_key, en_col_key in zip(calib_keys, energy_column_keys):
    calib_data = exp_data[calib_key]
    psd_report = calib_data[ExperimentDataKey.PSD_REPORT]
    for border_key, n_col_key in zip(border_keys, n_column_keys):
        borders = calib_data[border_key]
        print(borders)
    
        fig, ax = plot_classification(
            psd_report,
            borders,
            experiment_id,
            n_col_key,
            en_col_key,
            count_limit=COUNT_LIMIT,
            colormap_name="seismic"
        )

        print(f"{calib_key.value}/{border_key.value}")
        print(borders.left)
        plt.show()

In [ ]:
# Save window boundaries
save_folder = INPUT_DATA_FOLDER / "ReferenceWindow"
save_folder.mkdir(parents=True, exist_ok=True)

calib_keys = [ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
energy_column_keys = [DetectorDataframeColumn.CALIB_ENERGY, DetectorDataframeColumn.RECALIBRATED_ENERGY]
border_keys = [
    ExperimentDataKey.NASA_BORDERS,
    ExperimentDataKey.NASA_BORDERS_RECALC,
    ExperimentDataKey.N_WINDOW_BORDERS
]
n_column_keys = [
    DetectorDataframeColumn.NEUTRON_CLASS, 
    DetectorDataframeColumn.NEUTRON_RECALC_CLASS, 
    DetectorDataframeColumn.NEW_N_CLASS
]

for calib_key in calib_keys:
    calib_data = exp_data[calib_key]
    for border_key in border_keys:
        borders = calib_data[border_key]
    
        left_border = borders.left
        right_border = borders.right
        # TODO put calibration type in save file path
        side_borders_file_path = save_folder / f"{calib_key.value}_{border_key.value}_side_borders.txt"
        with side_borders_file_path.open('w') as side_file:
            side_file.writelines(
                [
                    f"left: {left_border if left_border is not None else 'None'}",
                    f"right: {right_border if right_border is not None else 'None'}"
                ]
            )
        print(f"Saved side borders to {side_borders_file_path}")
        
        bottom_border = borders.bottom
        if bottom_border is not None:
            bottom_border_file_path = save_folder / f"{calib_key.value}_{border_key.value}_bottom_border.pkl"
            with bottom_border_file_path.open('wb') as bottom_file:
                pickle.dump(bottom_border, bottom_file)
            print(f"Saved bottom border to {bottom_border_file_path}")
        else:
            print("No bottom border")
    
        top_border = borders.top
        if top_border is not None:
            top_border_file_path = save_folder / f"{calib_key.value}_{border_key.value}_top_border.pkl"
            with top_border_file_path.open('wb') as top_file:
                pickle.dump(top_border, top_file)
            print(f"Saved top border to {top_border_file_path}")
        else:
            print("No top border")

In [ ]:
input("Processing done, hit Enter to finish")
stop()